# Foundation Model Comparison

This notebook reproduces the foundation-model comparison summary from cached prediction tables. It does not retrain models or reload embeddings. The source prediction tables are `Crick_early.csv` and `Crick_41.csv` under `paper/2_Foundation_model_comparison/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


MODEL_COLUMNS = {
    "LucaVirus": "prediction_lucavirus",
    "LucaOne": "prediction_lucaone",
    "ESM3B": "prediction_esm3b",
}
DATASETS = {
    "Internal test": "Crick_early.csv",
    "External season 41": "Crick_41.csv",
}
SUBSETS = ["All", "H1N1", "H3N2"]
METRICS = ["MAE", "MSE", "Pearson", "Spearman", "R2"]


def find_project_root(start=None):
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / "paper" / "2_Foundation_model_comparison").exists():
            return path
    raise RuntimeError(f"Cannot find project root from: {current}")


project_root = find_project_root()
source_dir = project_root / "paper" / "2_Foundation_model_comparison"
table_dir = project_root / "paper" / "Table"
figure_dir = project_root / "paper" / "Figure"

summary_csv = table_dir / "foundation_model_comparison_metrics.csv"
figure_png = figure_dir / "foundation_model_comparison.png"
figure_svg = figure_dir / "foundation_model_comparison.svg"


def load_prediction_tables():
    tables = {}
    for dataset_name, filename in DATASETS.items():
        path = source_dir / filename
        if not path.exists():
            raise FileNotFoundError(path)
        tables[dataset_name] = pd.read_csv(path)
    return tables


def r2_score_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return np.nan if ss_tot == 0 else 1 - ss_res / ss_tot


def compute_metrics(y_true, y_pred):
    values = pd.DataFrame({"observed": y_true, "predicted": y_pred}).dropna()
    observed = values["observed"].astype(float)
    predicted = values["predicted"].astype(float)
    error = observed - predicted

    return {
        "N": len(values),
        "MAE": np.mean(np.abs(error)),
        "MSE": np.mean(error ** 2),
        "Pearson": observed.corr(predicted, method="pearson"),
        "Spearman": observed.corr(predicted, method="spearman"),
        "R2": r2_score_np(observed, predicted),
    }


def subset_frame(df, subset):
    if subset == "All":
        return df
    return df[df["Type"] == subset]


def build_metric_table(tables):
    rows = []
    for dataset_name, df in tables.items():
        for subset in SUBSETS:
            subset_df = subset_frame(df, subset)
            for model_name, pred_col in MODEL_COLUMNS.items():
                metrics = compute_metrics(subset_df["label"], subset_df[pred_col])
                rows.append({
                    "Dataset": dataset_name,
                    "Subset": subset,
                    "Model": model_name,
                    **metrics,
                })
    return pd.DataFrame(rows)


def plot_overall_metrics(metric_table):
    overall = metric_table[metric_table["Subset"] == "All"].copy()
    model_order = list(MODEL_COLUMNS)
    dataset_order = list(DATASETS)
    colors = {"LucaVirus": "#d94b3d", "LucaOne": "#3f7fc1", "ESM3B": "#6f6f6f"}

    fig, axes = plt.subplots(2, 2, figsize=(8.0, 6.2), sharex=True)
    axes = axes.ravel()
    for ax, metric in zip(axes, ["MAE", "MSE", "Pearson", "Spearman"]):
        x = np.arange(len(dataset_order))
        width = 0.24
        for i, model_name in enumerate(model_order):
            values = [
                overall[(overall["Dataset"] == dataset) & (overall["Model"] == model_name)][metric].iloc[0]
                for dataset in dataset_order
            ]
            ax.bar(x + (i - 1) * width, values, width, label=model_name, color=colors[model_name])
        ax.set_title(metric)
        ax.set_xticks(x)
        ax.set_xticklabels(dataset_order, rotation=20, ha="right")
        ax.grid(axis="y", alpha=0.25)

    axes[0].set_ylabel("Lower is better")
    axes[2].set_ylabel("Higher is better")
    axes[0].legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(1.05, 1.32))
    fig.tight_layout()
    return fig


def save_outputs(metric_table, fig):
    table_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)
    metric_table.to_csv(summary_csv, index=False)
    fig.savefig(figure_png, dpi=600, bbox_inches="tight")
    fig.savefig(figure_svg, bbox_inches="tight")
    print(f"Saved: {summary_csv}")
    print(f"Saved: {figure_png}")
    print(f"Saved: {figure_svg}")

In [ ]:
tables = load_prediction_tables()
metric_table = build_metric_table(tables)
metric_table

In [ ]:
fig = plot_overall_metrics(metric_table)
save_outputs(metric_table, fig)
plt.show()